## Step 0 — Setup & model loading

Imports, repo root, FastF1 cache. `PitAgentCFG` loads five model artefacts — the
three N15 quantile regressors (P05/P50/P95 physical stop duration), the N16
undercut LightGBM classifier, and its Platt calibrator — plus the two JSON configs
that carry feature lists, thresholds, and the circuit traversal lookup.

Two aggregate lookup tables are computed at startup from the N16 training parquet
(`undercut_clean.parquet`): `circuit_undercut_rate` and `team_x_undercut_rate`.
These are pre-aggregated means that would otherwise be unavailable at inference time.
The N15 `team_year_median` feature — median physical stop duration per team × year —
defaults to a global constant of 2.8 s (center of the [2.0, 4.5 s] training range)
since the per-team breakdown was not exported separately; the impact on P50 accuracy
is marginal given it ranks low in N15's permutation importance.

Models to load:

| File | Source | Contents |
|---|---|---|
| `pit_prediction/hist_pit_p05_v1.pkl` | N15 | HistGBT P05 physical stop |
| `pit_prediction/hist_pit_p50_v1.pkl` | N15 | HistGBT P50 physical stop |
| `pit_prediction/hist_pit_p95_v1.pkl` | N15 | HistGBT P95 physical stop |
| `pit_prediction/model_config.json` | N15 | Features, circuit traversal lookup, team LabelEncoder classes |
| `pit_prediction/lgbm_undercut_v1.pkl` | N16 | LightGBM undercut classifier |
| `pit_prediction/calibrator_undercut_v1.pkl` | N16 | Platt calibrator (val 2024) |
| `pit_prediction/model_config_undercut_v1.json` | N16 | Features, threshold, dry compound list |


In [1]:
import json, sys
from dataclasses import dataclass, field
from pathlib import Path

import fastf1
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Module-level globals — populated by load_session() before tool invocation
LAPS: pd.DataFrame = pd.DataFrame()
SESSION_META: dict = {}

# Team name aliases — FastF1 2025 uses "Racing Bulls"; N15 encoder was trained on "RB"
_TEAM_ALIASES: dict[str, str] = {"Racing Bulls": "RB"}


c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class PitAgentCFG:
    """Runtime configuration for the Pit Strategy Agent.

    Loads three N15 quantile regressors (physical stop duration P05/P50/P95), the
    N16 undercut LightGBM classifier with its Platt calibrator, and both JSON
    configs. All models are loaded with joblib — required on Windows paths with
    non-ASCII characters that break LightGBM's native save_model.

    circuit_traversal maps GP name to the estimated pit lane traversal time (s);
    subtracting it from the total pit stop yields physical_stop_est, the N15 target.
    team_encoder is a sklearn LabelEncoder reconstructed from the classes stored in
    model_config.json — it converts team name strings to integer codes for N15.
    team_year_median is a per-team×year fallback constant (2.8 s) rather than a
    per-combination lookup because the N15 training medians were not exported; the
    feature has low permutation importance and the approximation is adequate for
    agent-level decision making.

    circuit_undercut_rate and team_x_undercut_rate are aggregated from the N16
    training parquet at startup so that undercut feature construction at inference
    time does not require re-loading the full dataset on every call.

    undercut_threshold (0.522) is the F1-optimal threshold from N16 Optuna tuning.
    dry_compounds lists the compound names for which the N16 model is valid; calls
    with INTERMEDIATE or WET are rejected early.
    """

    model_name: str = "local-model"
    team_year_median_fallback: float = 2.8  # global fallback — see docstring

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        # --- N15: pit duration models ---
        _pit_dir = root / "data" / "models" / "pit_prediction"
        self.pit_p05_model = joblib.load(_pit_dir / "hist_pit_p05_v1.pkl")
        self.pit_p50_model = joblib.load(_pit_dir / "hist_pit_p50_v1.pkl")
        self.pit_p95_model = joblib.load(_pit_dir / "hist_pit_p95_v1.pkl")

        with open(_pit_dir / "model_config.json") as f:
            pit_cfg = json.load(f)

        self.pit_features: list[str]       = pit_cfg["features"]
        self.circuit_traversal: dict       = pit_cfg["circuit_traversal_lookup"]

        # Reconstruct LabelEncoder for 'team' feature from saved class list
        le = LabelEncoder()
        le.classes_ = np.array(pit_cfg["label_encoder_classes"]["team"])
        self.team_encoder: LabelEncoder = le

        # --- N16: undercut model ---
        self.undercut_model      = joblib.load(_pit_dir / "lgbm_undercut_v1.pkl")
        self.undercut_calibrator = joblib.load(_pit_dir / "calibrator_undercut_v1.pkl")

        with open(_pit_dir / "model_config_undercut_v1.json") as f:
            uc_cfg = json.load(f)

        self.undercut_features: list[str]  = uc_cfg["features"]
        self.undercut_threshold: float     = uc_cfg["best_threshold"]
        self.dry_compounds: list[str]      = uc_cfg["dry_compounds"]

        # --- Aggregate lookups from N16 training parquet ---
        _uc_parquet = root / "data" / "processed" / "undercut_labeled" / "undercut_clean.parquet"
        _uc_df = pd.read_parquet(_uc_parquet)
        self.circuit_undercut_rate: dict = (
            _uc_df.groupby("GP_Name")["undercut_success"].mean().to_dict()
        )
        self.team_x_undercut_rate: dict = (
            _uc_df.groupby("Team_X")["undercut_success"].mean().to_dict()
        )


In [3]:
CFG = PitAgentCFG()

print(f"N15 P50 model      : {type(CFG.pit_p50_model).__name__}")
print(f"N15 features       ({len(CFG.pit_features)}): {CFG.pit_features}")
print(f"N15 team classes   ({len(CFG.team_encoder.classes_)}): {list(CFG.team_encoder.classes_)}")
print(f"Circuit traversal  : {len(CFG.circuit_traversal)} circuits | Bahrain={CFG.circuit_traversal.get('Bahrain Grand Prix', 'N/A'):.2f}s")
print()
print(f"N16 undercut model : {type(CFG.undercut_model).__name__}")
print(f"N16 features       ({len(CFG.undercut_features)}): {CFG.undercut_features}")
print(f"N16 threshold      : {CFG.undercut_threshold}")
print(f"Dry compounds      : {CFG.dry_compounds}")
print()
print(f"circuit_undercut_rate : {len(CFG.circuit_undercut_rate)} circuits | Bahrain={CFG.circuit_undercut_rate.get('Bahrain Grand Prix', 'N/A'):.3f}")
print(f"team_x_undercut_rate  : {len(CFG.team_x_undercut_rate)} teams")
print(f"team_year_median fallback : {CFG.team_year_median_fallback} s")
print(f"EXPORT_DIR         : {CFG.export_dir}")


N15 P50 model      : HistGradientBoostingRegressor
N15 features       (9): ['team', 'year', 'tyre_life_in', 'lap_number', 'compound_id', 'compound_change', 'under_sc', 'tight_pit_box', 'team_year_median']
N15 team classes   (12): ['Alfa Romeo', 'AlphaTauri', 'Alpine', 'Aston Martin', 'Ferrari', 'Haas F1 Team', 'Kick Sauber', 'McLaren', 'Mercedes', 'RB', 'Red Bull Racing', 'Williams']
Circuit traversal  : 24 circuits | Bahrain=22.67s

N16 undercut model : LGBMClassifier
N16 features       (13): ['pos_gap', 'Lap_gap', 'tyre_life_diff', 'TyreLife_X', 'TyreLife_Y', 'compound_x_id', 'compound_y_id', 'compound_delta', 'pit_delta_X', 'lap_race_pct', 'pos_X_before', 'circuit_undercut_rate', 'team_x_undercut_rate']
N16 threshold      : 0.522
Dry compounds      : ['HARD', 'MEDIUM', 'SOFT']

circuit_undercut_rate : 24 circuits | Bahrain=0.360
team_x_undercut_rate  : 13 teams
team_year_median fallback : 2.8 s
EXPORT_DIR         : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\dat

### Step 0 results

| Artefact | Type | Details |
|---|---|---|
| N15 P05/P50/P95 | HistGradientBoostingRegressor | 9 features — `team` LabelEncoded, `team_year_median` global fallback 2.8 s |
| N16 undercut | LGBMClassifier | 13 features, threshold=0.522, dry compounds only |
| N16 calibrator | LogisticRegression (Platt) | fitted val 2024 |
| circuit_traversal | dict | 24 circuits — Bahrain=22.67 s |
| circuit_undercut_rate | dict | 24 circuits — Bahrain=0.360 |
| team_x_undercut_rate | dict | 13 teams |

All five model artefacts loaded via joblib. The N15 team `LabelEncoder` is
reconstructed from the class list in `model_config.json` — no separate `.pkl`
needed. The two undercut rate lookups are computed once at startup from
`undercut_clean.parquet` (1 032 pairs, 2023–2024) and held in `CFG` for the
lifetime of the agent.


---

## Step 1 — `PitStrategyOutput` dataclass + feature helpers

Defines the agent's output structure and the feature engineering pipeline that
converts FastF1 lap data into the exact feature sets expected by N15 (9 features,
physical stop duration) and N16 (13 features, undercut success).

Feature construction replicates the N15/N16 training pipelines exactly — same
column names, same encodings, same aggregate lookups. Three helper functions cover
each responsibility independently: building N15 input, building N16 input, and
identifying undercut candidate rivals.

`compound_id` (N15) and `compound_x_id`/`compound_y_id` (N16) encode the actual
Pirelli compound number (C1–C5) rather than the color name. The conversion from
SOFT/MEDIUM/HARD uses `TIRE_COMPOUNDS` when available, falling back to a
color→id map anchored to the most common Pirelli allocation (C1=HARD, C3=MEDIUM,
C5=SOFT).


In [ ]:
@dataclass
class PitStrategyOutput:
    """Structured output of the Pit Strategy Agent for one driver at one lap.

    action summarises the agent's recommendation: PIT_NOW triggers an immediate
    stop, STAY_OUT defers, UNDERCUT means pit before the target rival to gain
    track position, OVERCUT means stay out to exploit fresh-tyre pace later, and
    REACTIVE_SC means pit under an active or imminent Safety Car window.

    stop_duration_p05/p50/p95 are the N15 quantile predictions for the physical
    stop time (jack-up to release, excluding pit lane traversal). Adding
    circuit_traversal from CFG gives the total pit lane time.

    undercut_prob and undercut_target are None when no dry-compound rival is
    within max_pos_gap positions or when conditions are wet.

    sc_reactive flags that the recommendation was triggered by a high SC
    probability passed in from N27, not by tyre or position logic alone.
    """

    action: str                    # PIT_NOW | STAY_OUT | UNDERCUT | OVERCUT | REACTIVE_SC
    recommended_lap: int | None    # None when action is STAY_OUT
    compound_recommendation: str   # SOFT | MEDIUM | HARD
    stop_duration_p05: float
    stop_duration_p50: float
    stop_duration_p95: float
    undercut_prob: float | None    # None if no valid rival in range
    undercut_target: str | None    # FastF1 driver abbreviation of the undercut target
    sc_reactive: bool              # True if recommendation driven by SC probability
    reasoning: str


In [ ]:

# Fallback color→compound_id when TIRE_COMPOUNDS has no entry for the circuit/year
_COMPOUND_FALLBACK: dict[str, int] = {"HARD": 1, "MEDIUM": 3, "SOFT": 5}


def _compound_to_id(compound: str, gp_name: str, year: int) -> int:
    """Convert SOFT/MEDIUM/HARD to Pirelli compound number (C1–C5).

    Looks up the per-race allocation from TIRE_COMPOUNDS (keyed as 'YYYY_GP Name').
    Falls back to _COMPOUND_FALLBACK anchored to the most common mid-season Pirelli
    allocation when the circuit/year is absent from the map.
    """
    key = f"{year}_{gp_name}"
    allocation = TIRE_COMPOUNDS.get(key, {})
    return allocation.get(compound, _COMPOUND_FALLBACK.get(compound, 3))


def _encode_team(team_raw: str) -> int:
    """Encode team name to integer using the N15 LabelEncoder stored in CFG.

    Applies _TEAM_ALIASES before encoding so FastF1 variants ('Racing Bulls')
    resolve to the training-time name ('RB'). Unknown teams encode to 0 rather
    than raising to keep inference robust when new teams appear mid-season.
    """
    team_str = _TEAM_ALIASES.get(team_raw, team_raw)
    try:
        return int(CFG.team_encoder.transform([team_str])[0])
    except ValueError:
        return 0


In [ ]:

def _get_lap_row(driver: str, lap_number: int) -> pd.Series | None:
    """Return the single LAPS row for driver at lap_number, or None if missing."""
    rows = LAPS[(LAPS["Driver"] == driver) & (LAPS["LapNumber"] == lap_number)]
    return rows.iloc[0] if not rows.empty else None


def _get_position_map(lap_number: int) -> dict[str, float]:
    """Return {driver: position} for all drivers at lap_number with valid position."""
    lap_data = LAPS[LAPS["LapNumber"] == lap_number][["Driver", "Position"]].dropna()
    return dict(zip(lap_data["Driver"], lap_data["Position"]))


def _compounds_are_dry(comp_x: str, comp_y: str) -> bool:
    """Return True only if both compounds are in CFG.dry_compounds.

    The N16 model was trained exclusively on dry-compound stints (HARD/MEDIUM/SOFT).
    Calling it on INTERMEDIATE or WET would extrapolate beyond the training
    distribution and produce unreliable probabilities.
    """
    return comp_x in CFG.dry_compounds and comp_y in CFG.dry_compounds


In [ ]:

def build_pit_duration_features(
    driver: str,
    lap_number: int,
    compound: str,
    compound_change: bool,
    under_sc: bool,
) -> pd.DataFrame:
    """Build the 9-feature input row for the N15 quantile regressors.

    Assembles features from LAPS, SESSION_META, and CFG lookups. team_year_median
    uses CFG.team_year_median_fallback (2.8 s global constant) because per-team×year
    medians were not exported from N15. tight_pit_box is hardcoded False — the
    feature was in N15 training but has near-zero permutation importance and is
    not reliably derivable from FastF1 at runtime. Raises ValueError if no lap
    row exists for the driver at the given lap.
    """
    row = _get_lap_row(driver, lap_number)
    if row is None:
        raise ValueError(f"No lap data for {driver} at lap {lap_number}")

    gp_name  = SESSION_META.get("gp_name", "")
    year     = SESSION_META.get("year", 2025)
    team_raw = SESSION_META.get("team_lookup", {}).get(driver, "Unknown")

    feat = {
        "team":             _encode_team(team_raw),
        "year":             year,
        "tyre_life_in":     int(row.get("TyreLife", 1)),
        "lap_number":       lap_number,
        "compound_id":      _compound_to_id(compound, gp_name, year),
        "compound_change":  int(compound_change),
        "under_sc":         int(under_sc),
        "tight_pit_box":    0,
        "team_year_median": CFG.team_year_median_fallback,
    }
    return pd.DataFrame([feat])[CFG.pit_features]


In [ ]:

def build_undercut_features(
    driver_x: str,
    driver_y: str,
    lap_number: int,
) -> pd.DataFrame | None:
    """Build the 13-feature input row for the N16 undercut classifier.

    Returns None if either driver has no lap row at lap_number or if either
    compound is not dry (wet/intermediate conditions are out of N16's training scope).
    compound_delta encodes relative hardness: negative means attacker is on a softer
    tyre. pit_delta_X estimates total stop cost as circuit_traversal + 4.5 s (a
    conservative physical stop median); it represents inlap + outlap minus two
    representative race laps and is the main determinant of whether a position gap
    can be bridged. circuit_undercut_rate and team_x_undercut_rate come from CFG
    lookups pre-aggregated at startup from the N16 training parquet.
    """
    x_row = _get_lap_row(driver_x, lap_number)
    y_row = _get_lap_row(driver_y, lap_number)
    if x_row is None or y_row is None:
        return None

    comp_x = str(x_row.get("Compound", "MEDIUM")).upper()
    comp_y = str(y_row.get("Compound", "MEDIUM")).upper()
    if not _compounds_are_dry(comp_x, comp_y):
        return None

    gp_name    = SESSION_META.get("gp_name", "")
    year       = SESSION_META.get("year", 2025)
    total_laps = SESSION_META.get("total_laps", 57)
    team_x_raw = SESSION_META.get("team_lookup", {}).get(driver_x, "Unknown")
    team_x     = _TEAM_ALIASES.get(team_x_raw, team_x_raw)

    comp_x_id = _compound_to_id(comp_x, gp_name, year)
    comp_y_id = _compound_to_id(comp_y, gp_name, year)

    feat = {
        "pos_gap":               float(y_row.get("Position", 9)) - float(x_row.get("Position", 10)),
        "Lap_gap":               lap_number,
        "tyre_life_diff":        float(x_row.get("TyreLife", 10)) - float(y_row.get("TyreLife", 10)),
        "TyreLife_X":            float(x_row.get("TyreLife", 10)),
        "TyreLife_Y":            float(y_row.get("TyreLife", 10)),
        "compound_x_id":         comp_x_id,
        "compound_y_id":         comp_y_id,
        "compound_delta":        comp_x_id - comp_y_id,
        "pit_delta_X":           CFG.circuit_traversal.get(gp_name, 20.0) + 4.5,
        "lap_race_pct":          lap_number / total_laps,
        "pos_X_before":          float(x_row.get("Position", 10)),
        "circuit_undercut_rate": CFG.circuit_undercut_rate.get(gp_name, 0.38),
        "team_x_undercut_rate":  CFG.team_x_undercut_rate.get(team_x, 0.38),
    }
    return pd.DataFrame([feat])[CFG.undercut_features]


In [ ]:

def get_undercut_candidates(
    driver: str,
    lap_number: int,
    max_pos_gap: int = 5,
) -> list[str]:
    """Return drivers ahead of driver within max_pos_gap positions at lap_number.

    Uses _get_position_map to fetch positions for all drivers at the lap, then
    filters to those strictly ahead (lower position number) and within the gap
    window. The result is sorted by position ascending so the immediate car ahead
    is first — the most likely undercut target for the agent to score first.
    """
    pos_map = _get_position_map(lap_number)
    my_pos  = pos_map.get(driver)
    if my_pos is None:
        return []

    candidates = [
        d for d, p in pos_map.items()
        if d != driver and (my_pos - max_pos_gap) <= p < my_pos
    ]
    return sorted(candidates, key=lambda d: pos_map[d])


In [ ]:

def smoke_test_step1():
    """Verify feature builders return correctly shaped DataFrames with expected columns."""
    feat_pit = build_pit_duration_features(
        driver="NOR", lap_number=18,
        compound="MEDIUM", compound_change=True, under_sc=False,
    )
    print("N15 feature row:")
    print(feat_pit.to_string())
    print()

    feat_uc = build_undercut_features("NOR", "PIA", lap_number=18)
    if feat_uc is not None:
        print("N16 feature row:")
        print(feat_uc.to_string())
    else:
        print("build_undercut_features → None (wet compound or missing lap data)")
    print()

    candidates = get_undercut_candidates("NOR", lap_number=18)
    print(f"Undercut candidates ahead of NOR @ lap 18: {candidates}")

smoke_test_step1()
